In [ ]:
import pandas as pd

# =========================================================
# 1. PATH SETUP
# =========================================================
FLOW_FILE                = "../data/intermediate/park_flows_placekey.csv"
TRACT_AGG_OUTPUT         = "../data/intermediate/tract_park_aggregation.csv"
PLACEKEY_AGG_OUTPUT      = "../data/intermediate/placekey_aggregation.csv"

# Optional distance restriction for placekey aggregation (set to None to disable)
MAX_DIST_KM = None


In [ ]:
# =========================================================
# 2. LOAD FLOW DATA
# =========================================================
print("Loading flow data...")
flow_df = pd.read_csv(FLOW_FILE)
flow_df.columns = flow_df.columns.str.strip()
print(f"Loaded {len(flow_df)} rows.")


In [ ]:
# =========================================================
# 3. TRACT-LEVEL AGGREGATION
# For each tract: total visits and list of [park, distance, visits] triplets
# =========================================================
print("Aggregating to tract level...")

flow_df['park_detail'] = flow_df.apply(
    lambda x: [x['park_j'], x['distance_km'], x['visits']],
    axis=1
)

tract_agg = flow_df.groupby('tract_i').agg(
    total_visits=('visits', 'sum'),
    park_visits=('park_detail', list)
).reset_index()

tract_agg['park_visits'] = tract_agg['park_visits'].astype(str)
tract_agg = tract_agg.sort_values('tract_i').reset_index(drop=True)
tract_agg.to_csv(TRACT_AGG_OUTPUT, index=False)
print(f"Tract aggregation complete: {len(tract_agg)} tracts -> {TRACT_AGG_OUTPUT}")
tract_agg.head()


In [ ]:
# =========================================================
# 4. PLACEKEY-LEVEL AGGREGATION
# For each park POI: total visits and list of visiting tracts
# Optional: restrict to flows within MAX_DIST_KM
# =========================================================
print("Aggregating to placekey level...")

pk_df = flow_df.copy()

if MAX_DIST_KM is not None:
    before = len(pk_df)
    pk_df = pk_df[pk_df['distance_km'] <= MAX_DIST_KM]
    print(f"Distance filter ({MAX_DIST_KM}km): {before} -> {len(pk_df)} rows")
else:
    print("No distance filter applied.")

pk_agg = pk_df.groupby('park_j').agg(
    total_visits=('visits', 'sum'),
    num_tracts=('tract_i', 'nunique'),
    tract_list=('tract_i', lambda x: list(x.unique()))
).reset_index()

pk_agg = pk_agg.rename(columns={'park_j': 'placekey'})
pk_agg['tract_list'] = pk_agg['tract_list'].astype(str)
pk_agg = pk_agg.sort_values('placekey').reset_index(drop=True)
pk_agg.to_csv(PLACEKEY_AGG_OUTPUT, index=False)
print(f"Placekey aggregation complete: {len(pk_agg)} POIs -> {PLACEKEY_AGG_OUTPUT}")
pk_agg.head()
